In [12]:
import librosa
import IPython.display as ipd
import requests
import io
import re
import boto3
from botocore import UNSIGNED
from botocore.client import Config

from google.cloud import storage

# Note
Mixer functions originally built for wav files but some of these are stored as flac / other types - may need to change mixer functions to sf.read() instead of wavfile.read()

# Step 1
Get URLs from event buckets of interest and store them in a dictionary that has a user-friendly key for the clip.  For NOAA SanctSound clips:

In [4]:
bucket_name = "noaa-passive-bioacoustic"
prefix = "sanctsound/products/sound_clips/"

client = storage.Client.create_anonymous_client()
bucket = client.bucket(bucket_name)
blobs = client.list_blobs(bucket, prefix=prefix)

urls = {}
keywords = ['shrimp', 'whale', 'ship', 'boat', 'dolphins', 'odontocete', 'wind', 'sealion', 'fish', 'ship', 'vessel', 'scuba', 'hurricane', 'pinniped', 'seal', 'bocaccio', 'sonar', 'rain']

for blob in blobs:
    if not blob.name.endswith('.wav'):
        continue

    match = re.search(r"SanctSound_([A-Za-z0-9]+_[A-Za-z0-9]+)_([^_]+)_\d{8}T\d{6}Z\.wav$", blob.name)
    if not match:
        continue

    site_id = match.group(1)
    event_name = match.group(2)

    if not any(k in event_name for k in keywords):
        continue
    
    combined_key = f"{event_name} - Soundscape {site_id}"

    public_url = f"https://storage.googleapis.com/{bucket_name}/{blob.name}"

    urls.setdefault(combined_key, []).append(public_url)

For OrcaSound clips:

In [17]:
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

response = s3.list_objects_v2(
    Bucket="acoustic-sandbox",
    Delimiter="/"
)

# folders
for p in response.get("CommonPrefixes", []):
    print("DIR ", p["Prefix"])

#for obj in response.get("Contents", []):
 #   print("FILE", obj["Key"], obj["Size"])

#paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket="acoustic-sandbox"):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith((".wav", ".flac")):
            print(obj["Key"])

DIR  2017-09-05-SRKW-highlight-hour/
DIR  2017-09-05-SRKW/
DIR  2017-09-27-OS-continuous-wavs/
DIR  2017-09-27_OS_SRKW-wav/
DIR  2017_8_VesselsAndWavS/
DIR  2018-sperm-whale-Yukusam/
DIR  2019-11-14_PT_SRKW_HLS/
DIR  2019-Orcasound-examples/
DIR  2020-06-26-SRKW-Lpod/
DIR  2021_9_12_OS_YearsBestVocalPassby/
DIR  OS_AIS/
DIR  acoustic-separation/
DIR  ambient-sound-analysis/
DIR  clap-model/
DIR  data-audio-raw/
DIR  festtest/
DIR  go-test/
DIR  humpbacks/
DIR  labeled-data/
DIR  machineLearningFile/
DIR  orcaal-dev/
DIR  orcahello/
DIR  orcasounds/
DIR  results/
DIR  vessel-image-AI/
DIR  wholistener/
2017-09-05-SRKW-highlight-hour/OS_9_27_2017_08_41_00_.wav
2017-09-05-SRKW-highlight-hour/OS_9_27_2017_08_51_00_.wav
2017-09-05-SRKW-highlight-hour/OS_9_27_2017_09_08_00_.wav
2017-09-05-SRKW-highlight-hour/OS_9_27_2017_09_13_00_.wav
2017-09-05-SRKW-highlight-hour/OS_9_27_2017_09_29_00_.wav
2017-09-05-SRKW-highlight-hour/OS_9_27_2017_09_40_00_.wav
2017-09-05-SRKW/LK_20170905_250kHz_SMRU/LK_

In [ ]:


prefix = 'rpi_sunset_bay/hls/'

directories, num_directories = [], []
files, buckets = [], []

# List objects in the specified bucket and prefix
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix, Delimiter='/')

# Print directory (pseudo-folder) names
if 'CommonPrefixes' in response:
    for prefix_info in response['CommonPrefixes']:
        directories.append(prefix_info['Prefix'])
        # Get the numeric part of the directory name
        num = prefix_info['Prefix'].split('/')[-2]  # Get the last directory name before the trailing '/'
        num_directories.append(int(num))

# Print file names
if 'Contents' in response:
    for obj in response['Contents']:
        files.append(obj['Key'])

# List all buckets inside selected node
paginator = s3_client.get_paginator('list_objects_v2')
operation_parameters = {
    'Bucket': bucket_name,
    'Prefix': prefix,
    'Delimiter': '/'
}

for page in paginator.paginate(**operation_parameters):
    if 'CommonPrefixes' in page:
            for prefix_info in page['CommonPrefixes']:
                dir_name = prefix_info['Prefix']
                dir_number = int(dir_name.split('/')[-2])
                buckets.append(dir_number)

# Step 2
Function that will load audio for a given key.
My thought is user could "add" individual files they want to use to their library since it would probably take too long to store all in local memory (even though they'd be stored numerically)

In [19]:
def load_audio(key):
    if key not in urls:
        raise ValueError(f"Key '{key}' not found in urls.")

    # Each key may have multiple URLs; take the first one unless specified
    full_url = urls[key][0]

    response = requests.get(full_url)
    response.raise_for_status()

    audio_bytes = io.BytesIO(response.content)
    data, sr = sf.read(audio_bytes)

    return data, sr

load_audio("windwaves - Soundscape SB03_13")

(array([0.00000000e+00, 1.52587891e-04, 3.96728516e-04, ...,
        4.27246094e-04, 9.15527344e-05, 3.05175781e-05], shape=(240001,)),
 48000)